In [24]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

TARGET = 'prediction'
ID_COL = 'patient_id'


In [25]:
# загружаем данные
train_path = 'train.csv'
test_path = 'test_features.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
train.head()

Train shape: (800, 19)
Test shape: (200, 18)


,patient_id,Возраст,Пол_мужской,ИМТ,Окружность_талии_см,САД_мм_рт_ст,ДАД_мм_рт_ст,Пульсовое_давление,Глюкоза_натощак_ммоль_л,HbA1c_%,ЛПНП_ммоль_л,ЛПВП_ммоль_л,Триглицериды_ммоль_л,СКФ_мл_мин,Курение,Физическая_активность_мин_нед,prediction,Статус_глюкозы,Доклинический_риск
0,1,41,1,24.24,81.04,113.19,62.54,50.65,4.595,4.65,NaN,NaN,1.379,115.52,0,165.8,0,0,0
1,4,53,0,19.84,73.44,119.75,71.59,48.15,4.431,5.08,2.268,2.022,1.467,93.39,0,248.1,0,0,0
2,5,53,1,24.28,82.06,127.07,70.34,56.73,4.997,5.04,2.459,1.325,0.967,88.41,0,230.5,0,0,0
3,6,60,1,25.19,91.89,116.98,82.00,34.98,5.491,5.00,3.700,0.722,1.505,91.06,1,132.5,1,0,0
4,7,59,1,24.45,84.07,129.12,82.70,46.42,6.456,4.80,3.960,1.406,2.274,89.74,1,177.4,1,2,0


In [26]:
# определяем признаки и целевую переменную
features = [c for c in test.columns if c != ID_COL]
X = train[features]
y = train[TARGET].astype(np.float32).to_numpy()

print('Number of features:', len(features))
print('Class distribution:')
print(train[TARGET].value_counts())

Number of features: 17
Class distribution:
prediction
0    655
1    145
Name: count, dtype: int64


In [27]:
class PatientFFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [28]:
def fit_model(X_train, y_train, epochs=100):
    model = PatientFFN(X_train.shape[1])

    dataset = torch.utils.data.TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    )
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=32, shuffle=True
    )

    # учитываем дисбаланс
    positives = max((y_train == 1).sum(), 1)
    negatives = (y_train == 0).sum()
    pos_weight = torch.tensor(
        [negatives / positives], dtype=torch.float32
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for _ in range(epochs):
        model.train()
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    return model

## Локальная проверка модели

In [29]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_tr = scaler.fit_transform(imputer.fit_transform(X_tr)).astype(np.float32)
X_val = scaler.transform(imputer.transform(X_val)).astype(np.float32)

validation_model = fit_model(X_tr, y_tr)
validation_model.eval()

with torch.no_grad():
    val_prob = torch.sigmoid(
        validation_model(torch.tensor(X_val, dtype=torch.float32))
    ).numpy().ravel()

val_pred = (val_prob >= 0.5).astype(int)

print(f'Validation accuracy: {accuracy_score(y_val, val_pred):.4f}')
print(f'Validation F1:       {f1_score(y_val, val_pred):.4f}')
print(f'Validation ROC-AUC:  {roc_auc_score(y_val, val_prob):.4f}')

Validation accuracy: 0.9500
Validation F1:       0.8571
Validation ROC-AUC:  0.9868


## Обучение на всех публичных данных и предсказания

In [30]:
final_imputer = SimpleImputer(strategy='median')
final_scaler = StandardScaler()

X_full = final_scaler.fit_transform(
    final_imputer.fit_transform(train[features])
).astype(np.float32)

X_test = final_scaler.transform(
    final_imputer.transform(test[features])
).astype(np.float32)

final_model = fit_model(X_full, y)
final_model.eval()

with torch.no_grad():
    test_prob = torch.sigmoid(
        final_model(torch.tensor(X_test, dtype=torch.float32))
    ).numpy().ravel()

predictions = (test_prob >= 0.5).astype(int)

In [31]:
# формируем требуемый output
output = pd.DataFrame({
    ID_COL: test[ID_COL].to_numpy(),
    TARGET: predictions
})

assert len(output) == len(test)
assert output[ID_COL].equals(test[ID_COL])
assert list(output.columns) == ['patient_id', 'prediction']
assert output['prediction'].notna().all()

output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'task1_predictions.csv')
output.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Rows: {len(output)}')
output.head(10)

Saved: outputs/task1_predictions.csv
Rows: 200


,patient_id,prediction
0,2,0
1,3,1
2,8,0
3,18,0
4,19,0
5,20,0
6,28,0
7,29,0
8,37,0
9,43,0
